# 🇮🇳 Brahmi v3 — Qwen3-1.7B + Token Injection + Capability SFT (Kaggle 2×T4)

**Self-healing, fully resumable.** All progress persists to the private HF repo
`eulogik/brahmi-v3-artifacts`. If a session dies or OOMs: **Runtime → Run all**
and it continues from where it stopped.

| Phase | What | Resume guard |
|---|---|---|
| 1 | Mine Devanagari n-grams vs **Qwen3** vocab → top-300 tokens (~30 min CPU) | `phase1` |
| 2 | Inject + mean-init embeddings → ~1 MB `embed_delta.safetensors` | `phase2` |
| 3 | Warmup new embedding rows only (3K steps, ~40 min) | `warmup_step` |
| 4 | Capability SFT — LoRA r32 DDP on 2×T4, OOM-shrink retries | checkpoints + heartbeat |
| 5 | Merge + upload `eulogik/Bharat-Tiny-LLM-v3` (fp16) + M4 handoff commands | `phase5` |
| DPO | Optional scaffold (`RUN_DPO`) | flag |

### Quota watchdog
- OOM → halves per-device batch, doubles accumulation, relaunches
- Attempt with **zero step progress** counts toward stall; ≥2 stalled attempts →
  state saved, session **killed** to stop burning GPU quota
- Hard deadline `MAX_SESSION_HOURS=10.5` (Kaggle cap is 12) → graceful stop + kill

### One-time setup
Attach a Kaggle secret named **HF_TOKEN** (write access), or paste it into the
CONFIG cell. Everything else is automatic.


In [ ]:
#@title 0 · CONFIG {display-mode:"form"}
HF_TOKEN = ""  # or attach Kaggle secret named HF_TOKEN
BASE_MODEL = "Qwen/Qwen3-1.7B-Base"
ARTIFACTS_REPO = "eulogik/brahmi-v3-artifacts"
OUT_REPO = "eulogik/Bharat-Tiny-LLM-v3"

CORPUS_CANDIDATES = [
    ("eulogik/Bharat-Tiny-LLM-v2", "train_gold_v3.jsonl.gz", "model"),
    ("eulogik/Bharat-Tiny-LLM-v2", "train_gold_v3.jsonl.gz", "dataset"),
]

# phase 1-2
BUDGET = 300
MINE_LINES = 12000
VAL_LINES_OFFSET = 300000
VAL_LINES_COUNT = 2000

# phase 3 warmup
WARMUP_STEPS = 3000
WARMUP_BS = 4
WARMUP_SEQ = 256
WARMUP_LR = 1e-3

# phase 4 SFT
MAX_SFT_ROWS = 50000
SFT_EPOCHS = 2
SFT_LR = 1e-4
SFT_PER_DEVICE_BS = 4
SFT_GRAD_ACCUM = 2
SFT_SEQ = 1024
MIN_PER_DEVICE_BS = 1
MAX_ATTEMPTS = 4
STALL_LIMIT = 4
MAX_SESSION_HOURS = 10.5

RUN_DPO = False

import os
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        HF_TOKEN = os.environ.get("HF_TOKEN", "")
assert HF_TOKEN, "Set HF_TOKEN in CONFIG or attach Kaggle secret 'HF_TOKEN'"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print("CONFIG ok | budget:", BUDGET, "| base:", BASE_MODEL)


In [ ]:
#@title 1 · Setup {display-mode:"form"}
%pip install -q -U "transformers>=4.53" peft accelerate datasets safetensors sentencepiece huggingface_hub 2>&1 | tail -1
# peft>=0.18 tries torchao LoRA dispatch; Kaggle ships torchao 0.10 which it
# rejects. We don't use torchao quant LoRA so remove it to avoid the clash.
%pip uninstall -y torchao 2>/dev/null; echo "torchao removed"

import json, math, os, re, signal, subprocess, sys, time, gzip, random
from collections import Counter
from pathlib import Path

import os as _os
_os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
from huggingface_hub import (
    HfApi, create_repo, hf_hub_download, list_repo_files,
    snapshot_download, upload_file, upload_folder,
)

WORK = Path("/kaggle/working")
WORK.mkdir(exist_ok=True)
api = HfApi(token=HF_TOKEN)

GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS)
assert torch.cuda.device_count() >= 1, "No GPU attached!"


def kill_session(reason=""):
    """Flush everything and hard-exit so Kaggle stops billing GPU time."""
    print(f"\n*** KILLING SESSION: {reason} ***\n", flush=True)
    try:
        sys.stdout.flush()
        sys.stderr.flush()
        time.sleep(2)
    finally:
        os.kill(os.getpid(), signal.SIGKILL)


In [ ]:
#@title 2 · State manager — persists to HF Hub, survives any session death {display-mode:"form"}
STATE_PATH = WORK / "state_v3.json"


def load_state():
    try:
        p = hf_hub_download(repo_id=ARTIFACTS_REPO, filename="state_v3.json", repo_type="model",
                            token=HF_TOKEN, local_dir=str(WORK), force_download=True)
        st = json.loads(Path(p).read_text())
        print("loaded remote state")
        return st
    except Exception as e:
        print(f"fresh run (no remote state: {type(e).__name__})")
        return {}


def save_state(st):
    STATE_PATH.write_text(json.dumps(st, indent=2))
    upload_file(path_or_fileobj=str(STATE_PATH), path_in_repo="state_v3.json",
                repo_id=ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN,
                commit_message="state update")
    flags = {k: v for k, v in st.items() if isinstance(v, (int, str, bool))}
    print("state saved:", flags)


def ensure_repos():
    create_repo(ARTIFACTS_REPO, private=True, exist_ok=True, token=HF_TOKEN)
    create_repo(OUT_REPO, private=True, exist_ok=True, token=HF_TOKEN)


def fetch_latest_sft_checkpoint(local_dir):
    """Download newest sft/checkpoint-N adapter from Hub. Returns local path or None."""
    try:
        files = list_repo_files(ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN)
    except Exception:
        return None
    steps = []
    for f in files:
        m = re.match(r"sft/checkpoint-(\d+)/adapter_model\.safetensors$", f)
        if m:
            steps.append(int(m.group(1)))
    if not steps:
        return None
    latest = max(steps)
    target = Path(local_dir) / f"sft/checkpoint-{latest}"
    if target.exists():
        return target
    snapshot_download(ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN,
                      local_dir=local_dir,
                      allow_patterns=[f"sft/checkpoint-{latest}/**"])
    if target.exists():
        print(f"fetched remote checkpoint step {latest}")
        return target
    return None


STATE = load_state()
ensure_repos()
print("phase status:", {k: STATE.get(k) for k in
      ("phase1", "phase2", "phase3", "sft_data", "sft_done", "phase5")})


In [ ]:
#@title 3 · PHASE 1 — Mine top-300 Devanagari subwords vs Qwen3 vocab (~30 min CPU) {display-mode:"form"}
from transformers import AutoTokenizer

CORPUS_PATH = WORK / "corpus" / "train_gold_v3.jsonl.gz"


def get_corpus_handle():
    if CORPUS_PATH.exists():
        return gzip.open(CORPUS_PATH, "rt", encoding="utf-8")
    for rid, fn, rtype in CORPUS_CANDIDATES:
        try:
            p = hf_hub_download(repo_id=rid, filename=fn, repo_type=rtype, token=HF_TOKEN,
                                local_dir=str(WORK / "corpus"))
            print("corpus source:", rid, rtype)
            return gzip.open(p, "rt", encoding="utf-8")
        except Exception as e:
            print("skip", rid, rtype, "->", type(e).__name__)
    raise RuntimeError("train_gold_v3.jsonl.gz not found in any candidate repo")


def deva_runs(text):
    runs, run = [], ""
    for ch in text:
        if "\u0900" <= ch <= "\u097f":
            run += ch
        elif run:
            if len(run) >= 2:
                runs.append(run)
            run = ""
    if len(run) >= 2:
        runs.append(run)
    return runs


def read_lines(n):
    lines = []
    with get_corpus_handle() as f:
        for i, line in enumerate(f):
            if i >= n:
                break
            lines.append(line)
    return lines


def ngram_counts(lines, max_n=6):
    cb = {n: Counter() for n in range(2, max_n + 1)}
    for line in lines:
        try:
            msgs = json.loads(line)["messages"]
        except Exception:
            continue
        for m in msgs:
            for run in deva_runs(m["content"]):
                for n in range(2, max_n + 1):
                    for i in range(len(run) - n + 1):
                        cb[n][run[i:i + n]] += 1
    return cb


def fertility(tok, texts):
    tc = sum(len(t) for t in texts)
    tt = sum(len(tok.encode(t)) for t in texts)
    return tt / tc * 1000 if tc else 0.0


def select_tokens(cb, budget):
    scored = []
    for n in range(2, 7):
        for ng, freq in cb[n].most_common(2000):
            ids = qtok.encode(ng)
            if len(ids) > 1:
                scored.append(((len(ids) - 1) * math.sqrt(freq), ng))
    scored.sort(key=lambda x: -x[0])
    chosen, seen = [], set()
    for _, ng in scored:
        if ng in seen:
            continue
        seen.add(ng)
        chosen.append(ng)
        if len(chosen) >= budget:
            break
    return chosen


if STATE.get("phase1"):
    TOKENS = STATE["tokens"]
    qtok = AutoTokenizer.from_pretrained(BASE_MODEL)
    print(f"phase1 done — {len(TOKENS)} tokens loaded")
else:
    t0 = time.time()
    qt = AutoTokenizer.from_pretrained(BASE_MODEL)
    qtok = qt
    mine_lines = read_lines(MINE_LINES)
    val_lines = []
    with get_corpus_handle() as f:
        for i, line in enumerate(f):
            if i < VAL_LINES_OFFSET:
                continue
            if i >= VAL_LINES_OFFSET + VAL_LINES_COUNT:
                break
            val_lines.append(line)

    val_deva = []
    for line in val_lines:
        try:
            for m in json.loads(line)["messages"]:
                val_deva.extend(deva_runs(m["content"]))
        except Exception:
            pass

    base_fert = fertility(qt, val_deva)
    print(f"baseline Devanagari fertility: {base_fert:.1f} tok/1000 chars "
          f"({len(val_deva)} runs, {time.time()-t0:.0f}s)")

    cb = ngram_counts(mine_lines)
    TOKENS = select_tokens(cb, BUDGET)
    print(f"selected {len(TOKENS)} tokens; top10:", TOKENS[:10])

    probe = AutoTokenizer.from_pretrained(BASE_MODEL)
    probe.add_tokens(TOKENS)
    new_fert = fertility(probe, val_deva)
    sanity_savings = (1 - new_fert / base_fert) * 100
    print(f"injected fertility: {new_fert:.1f} → savings {sanity_savings:.1f}%")

    STATE["phase1"] = True
    STATE["tokens"] = TOKENS
    STATE["fertility_base"] = round(base_fert, 1)
    STATE["fertility_injected"] = round(new_fert, 1)
    save_state(STATE)


In [ ]:
#@title 4 · PHASE 2 — Injection: tokenizer + mean-init embedding delta (~5 min) {display-mode:"form"}
import torch
from safetensors.torch import save_file, load_file

DELTA_PATH = WORK / "embed_delta.safetensors"
TOK_DIR = WORK / "v3_tokenizer"

if not STATE.get("phase2"):
    from transformers import AutoTokenizer, AutoModelForCausalLM

    TOKENS = STATE["tokens"]
    pre_ids = {t: qtok.encode(t) for t in TOKENS}

    vtok = AutoTokenizer.from_pretrained(BASE_MODEL)
    base_len = len(vtok)
    added = vtok.add_tokens(TOKENS)
    assert added == len(TOKENS), f"added {added} != {len(TOKENS)}"
    new_ids = list(range(base_len, base_len + added))
    tok_to_id = {t: i for t, i in zip(TOKENS, new_ids)}

    print("loading base model on CPU for mean-init...")
    base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16)
    emb = base_model.get_input_embeddings().weight.data
    rows = torch.zeros((len(TOKENS), emb.shape[1]), dtype=torch.float16)
    with torch.no_grad():
        for j, t in enumerate(TOKENS):
            const = torch.tensor(pre_ids[t], dtype=torch.long)
            rows[j] = emb[const].mean(dim=0).to(torch.float16)
    del base_model

    save_file({"ids": torch.tensor(new_ids, dtype=torch.int64),
               "rows": rows}, str(DELTA_PATH))
    vtok.save_pretrained(TOK_DIR)

    upload_file(path_or_fileobj=str(DELTA_PATH),
                path_in_repo="embed_delta.safetensors", repo_id=ARTIFACTS_REPO,
                repo_type="model", token=HF_TOKEN)
    for f in sorted(TOK_DIR.iterdir()):
        if f.is_file():
            upload_file(path_or_fileobj=str(f), path_in_repo=f"v3_tokenizer/{f.name}",
                        repo_id=ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN)
    STATE["phase2"] = True
    save_state(STATE)
    print(f"delta saved: {rows.shape} fp16, ids {new_ids[0]}..{new_ids[-1]}")
else:
    if not DELTA_PATH.exists():
        hf_hub_download(repo_id=ARTIFACTS_REPO, filename="embed_delta.safetensors",
                        repo_type="model", token=HF_TOKEN, local_dir=str(WORK))
    if not TOK_DIR.exists():
        snapshot_download(repo_id=ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN,
                          local_dir=str(WORK), allow_patterns=["v3_tokenizer/*"])
    print("phase2 artifacts fetched")

d = load_file(str(DELTA_PATH))
print("delta ready | new ids:", d["ids"].tolist()[:3], "... shape:", d["rows"].shape)


In [ ]:
#@title 5 · PHASE 3 — Warmup: train ONLY the 300 new embedding rows (~40 min GPU) {display-mode:"form"}
import torch
from torch.utils.data import DataLoader
from safetensors.torch import save_file, load_file

WARMUP_PATH = WORK / "warmup_rows.safetensors"

# resume mid-warmup: pull partial artifact saved by a previous session
if not WARMUP_PATH.exists() and STATE.get("warmup_step"):
    try:
        hf_hub_download(repo_id=ARTIFACTS_REPO, filename="warmup_rows.safetensors",
                        repo_type="model", token=HF_TOKEN, local_dir=str(WORK),
                        force_download=True)
        print(f"resuming warmup from step {STATE['warmup_step']}")
    except Exception as e:
        print("no remote warmup artifact yet:", type(e).__name__)

if not STATE.get("phase3"):
    from transformers import AutoModelForCausalLM

    tok = AutoTokenizer.from_pretrained(TOK_DIR)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL,
                                                 torch_dtype=torch.float16).cuda()

    d = load_file(str(DELTA_PATH))
    ids = d["ids"].tolist()
    assert ids == sorted(set(ids)) and len(ids) == BUDGET \
        and ids[-1] - ids[0] == BUDGET - 1, "delta ids must be unique & contiguous"
    first_id = ids[0]

    # NOTE: Qwen3-1.7B TIES embed_tokens & lm_head — resize keeps them tied,
    # so writing rows here makes new tokens both input-embeddable AND predictable.
    model.resize_token_embeddings(ids[-1] + 1)
    emb = model.get_input_embeddings()
    dev, dt = emb.weight.device, emb.weight.dtype
    with torch.no_grad():
        emb.weight[ids] = d["rows"].to(device=dev, dtype=dt)
        if WARMUP_PATH.exists():
            w = load_file(str(WARMUP_PATH))
            emb.weight[w["ids"].tolist()] = w["rows"].to(device=dev, dtype=dt)
            print("resumed warmup rows from artifact")

    for p in model.parameters():
        p.requires_grad_(False)
    emb.weight.requires_grad_(True)

    corpus = get_corpus_handle()
    id_set = set(ids)
    chunks = []
    for line in corpus:
        if len(chunks) >= 20000:
            break
        try:
            text = " ".join(m["content"] for m in json.loads(line)["messages"])[:4000]
        except Exception:
            continue
        tks = tok.encode(text)
        for i in range(0, len(tks) - WARMUP_SEQ, WARMUP_SEQ):
            ch = tks[i:i + WARMUP_SEQ + 1]
            if any(t in id_set for t in ch[:-1]):
                chunks.append(ch)
            if len(chunks) >= 20000:
                break
    corpus.close()
    print(f"warmup chunks: {len(chunks)}")

    def _collate(items):
        return torch.tensor(items, dtype=torch.long)

    # ---- poison gate: a resumed checkpoint from a NaN run is garbage ----
    probe = next(iter(DataLoader(chunks[:32], batch_size=8, shuffle=False,
                                 collate_fn=_collate)))
    px = probe[:, :-1].cuda()
    py = probe[:, 1:].cuda()
    model.eval()
    with torch.no_grad():
        pl = torch.nn.functional.cross_entropy(
            model(input_ids=px).logits.view(-1, model.config.vocab_size).float(),
            py.view(-1))
    model.train()
    if not torch.isfinite(pl):
        if STATE.get("warmup_step"):
            print(f"!! probe loss {pl.item()} non-finite -> DISCARDING poisoned "
                  f"warmup checkpoint (step {STATE['warmup_step']}), restarting clean")
            with torch.no_grad():
                emb.weight[ids] = d["rows"].to(device=emb.weight.device,
                                               dtype=emb.weight.dtype)
            WARMUP_PATH.unlink(missing_ok=True)
            STATE.pop("warmup_step", None)
            save_state(STATE)
        else:
            raise RuntimeError(f"probe loss non-finite on fresh init: {pl.item()}")
    else:
        print(f"probe loss finite: {pl.item():.3f}"
              + (f" (resuming from step {STATE.get('warmup_step')})"
                 if STATE.get('warmup_step') else " (fresh)"))

    # NOTE: AdamW on fp16 embeddings explodes (~constant-size steps vs 0.02-scale
    # rows -> overflow -> NaN within ~200 steps). SGD+momentum scales with grad.
    opt = torch.optim.SGD([emb.weight], lr=WARMUP_LR, momentum=0.9)

    dl = DataLoader(chunks, batch_size=WARMUP_BS, shuffle=True,
                    drop_last=True, collate_fn=_collate)

    start_step = STATE.get("warmup_step", 0)
    losses, step = [], start_step
    model.train()
    t0 = time.time()
    while step < WARMUP_STEPS:
        oom_streak = 0
        nan_skips = 0
        for batch in dl:
            x = batch[:, :-1].cuda(non_blocking=True)
            y = batch[:, 1:].cuda(non_blocking=True)
            try:
                logits = model(input_ids=x).logits
                loss = torch.nn.functional.cross_entropy(
                    logits.view(-1, logits.shape[-1]).float(), y.view(-1))
                loss.backward()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                oom_streak += 1
                if oom_streak > 25:
                    raise RuntimeError("warmup OOM: >25 consecutive skipped batches")
                continue
            oom_streak = 0
            if not torch.isfinite(loss):
                opt.zero_grad(set_to_none=True)
                nan_skips += 1
                if nan_skips > 50:
                    raise RuntimeError("warmup: >50 non-finite losses")
                continue
            # mask grads: only the 300 injected rows may move
            g = emb.weight.grad
            if g is not None:
                g[:first_id] = 0
                torch.nn.utils.clip_grad_norm_([emb.weight], 1.0)
                if not torch.isfinite(g).all():
                    opt.zero_grad(set_to_none=True)
                    continue
            opt.step()
            opt.zero_grad(set_to_none=True)
            losses.append(loss.item())
            step += 1
            if step % 200 == 0:
                recent = sum(losses[-200:]) / min(200, len(losses))
                rate = (step - start_step) / max(time.time() - t0, 1)
                print(f"step {step}/{WARMUP_STEPS} loss {recent:.3f} ({rate:.2f} it/s)",
                      flush=True)
            if step % 1000 == 0 or step == WARMUP_STEPS:
                with torch.no_grad():
                    rows = emb.weight[ids].detach().to(torch.float16).cpu()
                save_file({"ids": d["ids"], "rows": rows}, str(WARMUP_PATH))
                upload_file(path_or_fileobj=str(WARMUP_PATH),
                            path_in_repo="warmup_rows.safetensors",
                            repo_id=ARTIFACTS_REPO, repo_type="model",
                            token=HF_TOKEN)
                STATE["warmup_step"] = step
                save_state(STATE)
            if step >= WARMUP_STEPS:
                break

    final = sum(losses[-100:]) / max(1, min(100, len(losses)))
    STATE["phase3"] = True
    STATE["warmup_final_loss"] = round(final, 4)
    save_state(STATE)
else:
    print("phase3 done | final warmup loss:", STATE.get("warmup_final_loss"))


In [ ]:
%%writefile train_sft.py
"""Brahmi v3 SFT — LoRA on 2xT4 via torchrun. Resume-safe, heartbeat, OOM-friendly."""
import argparse, json, os, random, time
from pathlib import Path

import torch
import torch.distributed as dist
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from safetensors.torch import load_file
from transformers import (AutoModelForCausalLM, AutoTokenizer, Trainer,
                          TrainingArguments, TrainerCallback)


class Heartbeat(TrainerCallback):
    def __init__(self, path):
        self.path = Path(path)

    def on_log(self, args, state, control, logs=None, **kw):
        if state.is_world_process_zero and logs and "loss" in logs:
            self.path.write_text(json.dumps(
                {"step": state.global_step, "loss": logs.get("loss"),
                 "ts": time.time()}))


def find_local_ckpt(out):
    cks = [p for p in Path(out).glob("checkpoint-*") if p.is_dir()]
    return max(cks, key=lambda p: int(p.name.split("-")[-1])) if cks else None


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="sft_out")
    ap.add_argument("--tok-dir", required=True)
    ap.add_argument("--delta", required=True)
    ap.add_argument("--warmup-rows", default="")
    ap.add_argument("--bs", type=int, default=4)
    ap.add_argument("--accum", type=int, default=2)
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=1e-4)
    ap.add_argument("--seq", type=int, default=1024)
    ap.add_argument("--resume-dir", default="")
    ap.add_argument("--heartbeat", default="progress.json")
    args = ap.parse_args()

    rank = int(os.environ.get("RANK", 0))
    local = int(os.environ.get("LOCAL_RANK", 0))
    if "WORLD_SIZE" in os.environ:
        dist.init_process_group("nccl")
    torch.cuda.set_device(local)

    tok = AutoTokenizer.from_pretrained(args.tok_dir)
    if tok.chat_template is None:
        # Base tokenizers sometimes ship without a template — use ChatML (Qwen-style)
        tok.chat_template = (
            "{% for message in messages %}"
            "{{'<|im_start|>' + message['role'] + '\\n' + message['content'] + '<|im_end|>\\n'}}"
            "{% endfor %}"
            "{% if add_generation_prompt %}{{'<|im_start|>assistant\\n'}}{% endif %}"
        )
        if rank == 0:
            print("base tokenizer had no chat template -> applied ChatML fallback")

    model = AutoModelForCausalLM.from_pretrained(
        os.environ["BASE_MODEL"], torch_dtype=torch.float16,
        attn_implementation="sdpa")

    # inject: resize to fit real token ids, then write mean-init + warmup rows
    d = load_file(args.delta)
    ids = d["ids"].tolist()
    assert ids == sorted(set(ids)) and len(ids) == int(ids[-1] - ids[0] + 1), \
        "delta ids must be unique & contiguous"
    model.resize_token_embeddings(ids[-1] + 1)
    emb = model.get_input_embeddings()
    with torch.no_grad():
        _dev, _dt = emb.weight.device, emb.weight.dtype
        emb.weight[ids] = d["rows"].to(device=_dev, dtype=_dt)
        if args.warmup_rows and Path(args.warmup_rows).exists():
            w = load_file(args.warmup_rows)
            emb.weight[w["ids"].tolist()] = w["rows"].to(device=_dev, dtype=_dt)
            if rank == 0:
                print("warmup rows applied")
    for p in emb.parameters():
        p.requires_grad_(False)   # embeddings stay frozen in SFT

    lcfg = LoraConfig(
        r=32, lora_alpha=64, lora_dropout=0.05, bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
    )
    model = get_peft_model(model, lcfg)
    model.config.use_cache = False
    if rank == 0:
        model.print_trainable_parameters()

    ds = load_dataset("json", data_files=args.data)["train"]

    def encode(ex):
        msgs = ex["messages"]
        if isinstance(msgs, str):
            msgs = json.loads(msgs)
        if not isinstance(msgs, list) or len(msgs) < 2:
            return {"input_ids": [tok.eos_token_id] * 2,
                    "labels": [-100] * 2}
        full = tok.apply_chat_template(msgs, tokenize=False,
                                       add_generation_prompt=False)
        prompt = tok.apply_chat_template(msgs[:-1], tokenize=False,
                                         add_generation_prompt=True)
        fids = tok(full, add_special_tokens=False)["input_ids"][:args.seq]
        pids = tok(prompt, add_special_tokens=False)["input_ids"]
        labels = list(fids)
        cut = min(len(pids), len(fids))
        for i in range(cut):
            labels[i] = -100
        return {"input_ids": fids, "labels": labels}

    ds = ds.map(encode, remove_columns=ds.column_names, num_proc=2,
                desc="tokenize")
    ds = ds.filter(lambda ex: any(l != -100 for l in ex["labels"]),
                   num_proc=2)
    ds = ds.shuffle(seed=42)

    class Collator:
        def __call__(self, feats):
            maxlen = max(len(f["input_ids"]) for f in feats)
            pad_id = tok.pad_token_id or tok.eos_token_id or 0
            batch_ids, batch_labels, batch_mask = [], [], []
            for f in feats:
                n = len(f["input_ids"])
                padn = maxlen - n
                batch_ids.append(f["input_ids"] + [pad_id] * padn)
                batch_labels.append(f["labels"] + [-100] * padn)
                batch_mask.append([1] * n + [0] * padn)
            t = lambda x: torch.tensor(x, dtype=torch.long)
            return {"input_ids": t(batch_ids),
                    "labels": t(batch_labels),
                    "attention_mask": t(batch_mask)}

    resume = args.resume_dir or ""
    local_ck = find_local_ckpt(args.out)
    if not resume and local_ck:
        resume = str(local_ck)
    if rank == 0 and resume:
        print("resuming from:", resume)

    targs = TrainingArguments(
        output_dir=args.out,
        per_device_train_batch_size=args.bs,
        gradient_accumulation_steps=args.accum,
        num_train_epochs=args.epochs,
        learning_rate=args.lr,
        lr_scheduler_type="cosine",
        lr_scheduler_kwargs={"warmup_ratio": 0.03},
        logging_steps=25,
        save_steps=500,
        save_total_limit=2,
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch_fused",
        report_to=[],
        dataloader_num_workers=2,
        seed=42,
        remove_unused_columns=False,
    )

    trainer = Trainer(model=model, args=targs, train_dataset=ds,
                      data_collator=Collator(),
                      callbacks=[Heartbeat(args.heartbeat)])
    trainer.train(resume_from_checkpoint=resume or None)

    if rank == 0:
        final = Path(args.out) / "final_adapter"
        trainer.model.save_pretrained(final)
        tok.save_pretrained(Path(args.out) / "tokenizer")
        (Path(args.out) / "DONE").write_text("ok")
        print("SFT complete ->", final)


if __name__ == "__main__":
    import traceback, sys
    _err_log = Path(os.environ.get("TRAIN_ERROR_LOG", "train_error.log"))
    try:
        main()
    except Exception:
        with open(_err_log, "w") as _f:
            traceback.print_exc(file=_f)
        traceback.print_exc()
        sys.stdout.flush()
        sys.stderr.flush()
        raise


In [ ]:
#@title 6a · PHASE 4 — Build capability SFT mix (~50K conversations) {display-mode:"form"}
SFT_DATA = WORK / "sft_data.jsonl.gz"

def norm_messages(rec):
    """Normalize common instruction schemas -> {'messages': [...]}. None if unusable."""
    try:
        if "messages" in rec and isinstance(rec["messages"], list):
            msgs = [{"role": m.get("role", "user"),
                     "content": str(m.get("content", ""))} for m in rec["messages"]]
        elif "instruction" in rec:
            user = rec["instruction"] + ("\n" + rec["input"] if rec.get("input") else "")
            msgs = [{"role": "user", "content": str(user)},
                    {"role": "assistant", "content": str(rec.get("output", ""))}]
        elif "question" in rec and "answer" in rec:
            msgs = [{"role": "user", "content": str(rec["question"])},
                    {"role": "assistant", "content": str(rec["answer"])}]
        elif "prompt" in rec and "response" in rec:
            msgs = [{"role": "user", "content": str(rec["prompt"])},
                    {"role": "assistant", "content": str(rec["response"])}]
        else:
            return None
    except Exception:
        return None
    if len(msgs) >= 2 and msgs[-1]["role"] == "assistant":
        if all(len(m["content"].strip()) > 1 for m in msgs):
            return {"messages": msgs}
    return None


def gsm_answer_fix(ans):
    import re as _re
    body = _re.sub(r"<<[^>]*>>", "", ans)
    m = _re.search(r"#{3,4}\s*(-?[\d,]+(?:\.\d+)?)", body)
    if m:
        final = m.group(1).replace(",", "")
        body = body.split("####")[0].split("###")[0].rstrip()
        return body.strip() + f"\nउत्तर: {final}"
    return body.strip()


if STATE.get("sft_data") and SFT_DATA.exists():
    print("sft_data ready")
else:
    from datasets import load_dataset
    random.seed(42)
    out, seen = [], set()

    def add(msgs, src):
        key = hash(json.dumps(msgs)[:400])
        if key not in seen:
            seen.add(key)
            out.append(msgs)

    # 1) GSM8K-Hindi train — teaches math CoT + 'उत्तर:' marker format
    gsm = load_dataset("bingbangboom/gsm8k-hindi")["train_main"]
    for r in gsm:
        ans = gsm_answer_fix(r["answer"])
        add([{"role": "user", "content": r["question"].strip()},
             {"role": "assistant", "content": ans}], "gsm")
    print("after gsm:", len(out))

    # 2) IndicVault (Hindi+Hinglish QA) — knowledge breadth, streamed subset
    try:
        iv = load_dataset("maya-research/IndicVault", name="Hindi",
                          split="train", streaming=True)
        want = min(30000, MAX_SFT_ROWS - len(out))
        got = 0
        for r in iv:
            if got >= want or len(out) >= MAX_SFT_ROWS:
                break
            m = norm_messages(r)
            if m:
                add(m, "iv")
                got += 1
    except Exception as e:
        print("IndicVault skipped:", type(e).__name__, e)
    print("after indicvault:", len(out))

    # 3) Hinglish instruct + govt schemes + recipes
    for rid, split in [("Sujalvc/hinglish-instruct-dataset", "train"),
                       ("Subh24ai/yojana-sahayak-instruct", "train"),
                       ("VishalMysore/cookGPT", "train")]:
        if len(out) >= MAX_SFT_ROWS:
            break
        try:
            dsi = load_dataset(rid, split=split)
            for r in dsi:
                if len(out) >= MAX_SFT_ROWS:
                    break
                m = norm_messages(r)
                if m:
                    add(m, rid)
        except Exception as e:
            print(rid, "skipped:", type(e).__name__)
    print("total:", len(out))

    random.shuffle(out)
    import gzip as _gz
    with _gz.open(SFT_DATA, "wt", encoding="utf-8") as f:
        for m in out:
            f.write(json.dumps({"messages": m}) + "\n")
    upload_file(path_or_fileobj=str(SFT_DATA),
                path_in_repo="sft_data.jsonl.gz", repo_id=ARTIFACTS_REPO,
                repo_type="model", token=HF_TOKEN)
    STATE["sft_data"] = True
    STATE["sft_rows"] = len(out)
    save_state(STATE)


## Phase 4b — Self-healing runner
Launches `train_sft.py` under torchrun (DDP 2×T4). OOM → shrink batch & retry. No progress across attempts → kills session to save quota.


In [ ]:
#@title 6b · Self-healing SFT runner {display-mode:"form"}

LOG_PATH = WORK / "sft_train.log"
HEARTBEAT = WORK / "progress.json"
DEADLINE = time.time() + MAX_SESSION_HOURS * 3600


def tail(path, n=1800):
    try:
        return Path(path).read_text(errors="ignore")[-n:]
    except Exception:
        return ""


def heartbeat_step():
    try:
        return json.loads(HEARTBEAT.read_text()).get("step", -1)
    except Exception:
        return -1


if not STATE.get("sft_done"):
    resume_dir = fetch_latest_sft_checkpoint(WORK / "resume")
    prev_best = heartbeat_step() if HEARTBEAT.exists() else -1
    bs, accum = SFT_PER_DEVICE_BS, SFT_GRAD_ACCUM
    stall = 0

    for attempt in range(1, MAX_ATTEMPTS + 1):
        if time.time() > DEADLINE:
            kill_session(f"session deadline hit before attempt {attempt}")
        print(f"\n=== SFT attempt {attempt}/{MAX_ATTEMPTS} | bs={bs} accum={accum} "
              f"| resume={resume_dir} ===", flush=True)

        cmd = ["torchrun", "--nproc_per_node=2", "--master_port=29515",
               "--log-dir", str(WORK / "torchrun_logs"),
               "train_sft.py",
               "--data", str(SFT_DATA), "--out", str(WORK / "sft_out"),
               "--tok-dir", str(TOK_DIR), "--delta", str(DELTA_PATH),
               "--warmup-rows", str(WARMUP_PATH),
               "--bs", str(bs), "--accum", str(accum),
               "--epochs", str(SFT_EPOCHS), "--lr", str(SFT_LR),
               "--seq", str(SFT_SEQ)]
        if resume_dir:
            cmd += ["--resume-dir", str(resume_dir)]

        env = dict(os.environ, BASE_MODEL=BASE_MODEL, HF_TOKEN=HF_TOKEN,
                    PYTHONFAULTHANDLER="1", PYTHONUNBUFFERED="1",
                    TRAIN_ERROR_LOG=str(WORK / "train_error.log"))
        stderr_path = WORK / "sft_train_stderr.log"
        with open(LOG_PATH, "w") as lf, open(stderr_path, "w") as ef:
            proc = subprocess.Popen(cmd, stdout=lf, stderr=ef,
                                    cwd=str(WORK), env=env)
            start_best = max(prev_best, 0)
            while proc.poll() is None:
                time.sleep(30)
                hs = heartbeat_step()
                if hs > prev_best:
                    prev_best = hs
                    print(f"  [hb] step={hs}", flush=True)
                if time.time() > DEADLINE:
                    print("deadline exceeded mid-run; terminating")
                    proc.terminate()
                    proc.wait(120)
                    break

        rc = proc.poll()
        done_flag = (WORK / "sft_out" / "DONE").exists()
        logtail = tail(LOG_PATH)

        if rc == 0 and done_flag:
            print("SFT finished successfully")
            break

        oom = ("OutOfMemoryError" in logtail) or ("CUDA out of memory" in logtail)
        progressed = prev_best > start_best
        stall = stall + (0 if progressed else 1)

        print(f"attempt {attempt}: rc={rc} done={done_flag} "
              f"oom={oom} progressed={progressed} stall={stall}")
        if rc != 0:
            print("--- torchrun log tail (last 800 chars) ---")
            print(logtail[-800:])
            print("--- end log ---")
            # Also print per-worker logs from log-dir
            for p in (WORK / "torchrun_logs").rglob("*.log"):
                try:
                    txt = p.read_text(errors="ignore")[-1200:]
                    print(f"--- worker log {p.name} ---")
                    print(txt)
                    print(f"--- end worker {p.name} ---")
                except Exception:
                    pass
            # Print raw stderr (the actual Python traceback)
            try:
                stderr_txt = stderr_path.read_text(errors="ignore")[-2000:]
                if stderr_txt.strip():
                    print("--- raw stderr ---")
                    print(stderr_txt)
                    print("--- end stderr ---")
            except Exception:
                pass
            # Print error log written by train_sft.py itself (most reliable)
            err_log = WORK / "train_error.log"
            if err_log.exists():
                print("--- train_error.log ---")
                print(err_log.read_text(errors="ignore")[-2000:])
                print("--- end train_error.log ---")

        if oom and bs > MIN_PER_DEVICE_BS:
            bs = max(MIN_PER_DEVICE_BS, bs // 2)
            accum *= 2
            print(f"OOM -> shrinking to bs={bs}, accum={accum}")
        elif not progressed:
            print(f"NO step progress this attempt ({stall}/{STALL_LIMIT})")

        resume_dir = fetch_latest_sft_checkpoint(WORK / "resume")

        if stall >= STALL_LIMIT:
            kill_session(f"training stalled: {stall} attempts with no progress")
    else:
        kill_session(f"max attempts ({MAX_ATTEMPTS}) exhausted")

    final_dir = WORK / "sft_out" / "final_adapter"
    assert final_dir.exists(), "no final adapter after successful run"
    upload_folder(folder_path=str(final_dir), repo_id=ARTIFACTS_REPO,
                  path_in_repo="sft/final", repo_type="model", token=HF_TOKEN)
    tokdir = WORK / "sft_out" / "tokenizer"
    for f in tokdir.iterdir():
        if f.is_file():
            upload_file(path_or_fileobj=str(f),
                        path_in_repo=f"sft/final_tokenizer/{f.name}",
                        repo_id=ARTIFACTS_REPO, repo_type="model", token=HF_TOKEN)
    STATE["sft_done"] = True
    save_state(STATE)
else:
    print("SFT already done")


In [ ]:
#@title 7 · PHASE 5 — Merge LoRA + upload Bharat-Tiny-LLM-v3 (~10 min) {display-mode:"form"}
if not STATE.get("phase5"):
    import torch
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from safetensors.torch import load_file

    print("loading base fp32 on CPU for exact merge...")
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)

    d = load_file(str(DELTA_PATH))
    ids = d["ids"].tolist()
    assert ids == sorted(set(ids)) and len(ids) == int(ids[-1] - ids[0] + 1), \
        "delta ids must be unique & contiguous"
    model.resize_token_embeddings(ids[-1] + 1)
    emb = model.get_input_embeddings()
    with torch.no_grad():
        emb.weight[ids] = d["rows"].float()
        if WARMUP_PATH.exists():
            w = load_file(str(WARMUP_PATH))
            emb.weight[w["ids"].tolist()] = w["rows"].float()
            print("warmup rows merged")

    model = PeftModel.from_pretrained(model, str(WORK / "sft_out" / "final_adapter"))
    model = model.merge_and_unload()
    model.half()

    EXPORT_DIR = WORK / "export"
    model.save_pretrained(str(EXPORT_DIR), safe_serialization=True,
                          max_shard_size="4GB")
    tok = AutoTokenizer.from_pretrained(TOK_DIR)
    tok.save_pretrained(str(EXPORT_DIR))

    upload_folder(folder_path=str(EXPORT_DIR), repo_id=OUT_REPO,
                  repo_type="model", token=HF_TOKEN,
                  commit_message=f"v3: Qwen3-1.7B + Brahmi-{BUDGET} + capability SFT")
    STATE["phase5"] = True
    save_state(STATE)
    del model
else:
    print("phase5 done — model live at:", OUT_REPO)

print(f"""
=== M4-side next steps (MLX quantize + benchmarks; cannot run on T4) ===

# 1. quantize to 4-bit MLX (on the Mac):
python -m mlx_lm convert --hf-path {OUT_REPO} \\
    -q --q-bits 4 --q-group-size 64 \\
    --mlx-path "/Volumes/KIOXIA 1TB/brahmi-v3-q4"

# 2. point benchmarks at v3 (edit V2_MODEL/V2_ADAPTER constants):
#    scripts/bench_fertility.py   -> fertility vs Qwen3 tokenizer
#    scripts/bench_ppl.py         -> bits/char fair comparison
#    scripts/bench_gsm_hi.py      -> does SFT fix the 0% math score?
""")


In [ ]:
#@title 8 · OPTIONAL — DPO polish (RUN_DPO flag in CONFIG; default off) {display-mode:"form"}
if not RUN_DPO:
    print("RUN_DPO=False -> skipping. Enable after SFT quality review.")
else:
    # Expects dpo_pairs.jsonl.gz on the artifacts repo:
    #   {"prompt": [...messages...], "chosen": "...", "rejected": "..."}
    %pip install -q trl 2>&1 | tail -1
    from trl import DPOTrainer, DPOConfig
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from safetensors.torch import load_file

    hf_hub_download(repo_id=ARTIFACTS_REPO, filename="dpo_pairs.jsonl.gz",
                    repo_type="model", token=HF_TOKEN, local_dir=str(WORK))
    pairs = load_dataset("json", data_files=str(WORK / "dpo_pairs.jsonl.gz"))["train"]

    import torch
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16).cuda()
    d = load_file(str(DELTA_PATH))
    ids = d["ids"].tolist()
    model.resize_token_embeddings(max(ids) + 1)
    _emb = model.get_input_embeddings()
    with torch.no_grad():
        _emb.weight[ids] = d["rows"].to(device=_emb.weight.device,
                                        dtype=_emb.weight.dtype)
    tokd = AutoTokenizer.from_pretrained(TOK_DIR)

    cfg = DPOConfig(
        output_dir=str(WORK / "dpo_out"), per_device_train_batch_size=2,
        gradient_accumulation_steps=8, beta=0.1, learning_rate=5e-7,
        num_train_epochs=1, logging_steps=25, save_steps=500,
        fp16=True, report_to=[],
    )
    dtrainer = DPOTrainer(model=model, args=cfg,
                          train_dataset=pairs, processing_class=tokd)
    dtrainer.train()
    dtrainer.save_model(str(WORK / "dpo_final"))
    upload_folder(folder_path=str(WORK / "dpo_final"), repo_id=ARTIFACTS_REPO,
                  path_in_repo="dpo/final", repo_type="model", token=HF_TOKEN)
    print("DPO done -> re-run PHASE 5 to export merged model")


In [ ]:
#@title 9 · Final status {display-mode:"form"}
import json, time
print(json.dumps({k: v for k, v in STATE.items() if k != "tokens"},
      indent=2, default=str))
print(f"\ntokens injected: {len(STATE.get('tokens', []))}")
print(f"artifacts repo : https://huggingface.co/{ARTIFACTS_REPO} (private)")
print(f"model repo     : https://huggingface.co/{OUT_REPO} (private until release)")
elapsed_h = 0
print("\nIf the session died mid-way: just Runtime > Run all. "
      "Completed phases are skipped automatically.")
